## Export Variant Video Library

- pull these functions into the main library later

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
from dask.distributed import wait

import scipy.stats
from sklearn.linear_model import LinearRegression

from matplotlib import pyplot as plt
import holoviews as hv
hv.extension('bokeh')

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

In [ ]:
import h5py
import cv2
import skimage as sk
from skimage.color import gray2rgb
import copy


def lookups_to_output_arr(headpath,x):
    for i in range(len(x)):
        with h5py.File(headpath + "/kymograph/kymograph_" + str(x.iloc[i]["File Index"]) +".hdf5","r") as infile:
            if i == 0:
                output_arr_dict = {channel:[] for channel in list(infile.keys())}
            for channel in list(infile.keys()):
                if x.iloc[i]["lane orientation"] == "top":
                    output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]][:,::-1])
                else:
                    output_arr_dict[channel].append(infile[channel][x.iloc[i]["File Trench Index"]])
    for channel in list(output_arr_dict.keys()):
        output_arr_dict[channel] = np.concatenate(output_arr_dict[channel],axis=2)
    return output_arr_dict

def export_avi(output_arr,outpath,min_intensity = 0,max_intensity = 10000, fps=10):
    output_arr_scaled = sk.exposure.rescale_intensity(output_arr,in_range=(min_intensity,max_intensity))
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(outpath,fourcc, fps, (output_arr_scaled.shape[2],output_arr_scaled.shape[1]))
    for t in range(output_arr_scaled.shape[0]):
        out.write(sk.color.gray2rgb(sk.util.img_as_ubyte(output_arr_scaled[t])))
    out.release()
    
def lookup_to_avi(headpath_list,outpath,x):
    tr.writedir(outpath, overwrite=False)
    
    output_arr_dict_list = []
    for i,headpath in enumerate(headpath_list):
        filtered_x = x[x["Experiment #"]==i]
        if len(filtered_x)>0:
            output_arr_dict = lookups_to_output_arr(headpath,filtered_x)
            output_arr_dict_list.append(output_arr_dict)
        
    merged_output_arr_dict = {}
    for channel in list(output_arr_dict.keys()):
        merged_output_arr_dict[channel] = np.concatenate([output_arr_dict[channel] for output_arr_dict in output_arr_dict_list], axis=2)
    del output_arr_dict_list
        
    for channel in list(merged_output_arr_dict.keys()):
        tr.writedir(outpath + "/" + channel, overwrite=False)
        export_avi(merged_output_arr_dict[channel],outpath + "/" + channel + "/" + str(x.iloc[0]["oDEPool7_id Temp"]) + ".avi")
    return True

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=200,
    n_workers_min=50,
    memory="4GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

In [ ]:
output_phenotype_kmyo_df_path_list = ["/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/"]
output_phenotype_kmyo_df_list = [dd.read_parquet(item,engine="pyarrow",calculate_divisions=True) for item in output_phenotype_kmyo_df_path_list]

# trench_to_lookup_df_list = []
variant_to_trench_df_list = []

for i in range(len(output_phenotype_kmyo_df_list)):
    trench_to_lookup_df = output_phenotype_kmyo_df_list[i].set_index("phenotype trenchid",sorted=True).groupby('phenotype trenchid').apply(lambda x: x.iloc[0])
    trench_to_lookup_df = trench_to_lookup_df[['File Index','File Trench Index','lane orientation','oDEPool7_id']].compute()
    trench_to_lookup_df["oDEPool7_id Temp"] = copy.deepcopy(trench_to_lookup_df["oDEPool7_id"])
    trench_to_lookup_df["Experiment #"] = i
    variant_to_trench_df = trench_to_lookup_df.reset_index().set_index("oDEPool7_id")
    variant_to_trench_df_list.append(variant_to_trench_df)
    
variant_to_trench_df = pd.concat(variant_to_trench_df_list,axis=0).sort_index()
variant_to_trench_dd = dd.from_pandas(variant_to_trench_df,npartitions=200).persist()

In [ ]:
nanopore_df = pd.read_csv('/home/lag36/scratch/lag36/2022-09-01_lDE24_nanopore/2021-04-20_Essential_KO_Lib_df_coords_corrected.tsv', sep='\t')

In [ ]:
nanopore_df[nanopore_df['Gene']=='nagE']

In [ ]:
headpath_list = ["/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/"]
output_path = '/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/AVI_Library/'

variant_to_trench_dd.groupby("oDEPool7_id").apply(lambda x: lookup_to_avi(headpath_list,output_path,x), meta=bool).compute()

In [ ]:
#check that number of images agrees with merged df when export done

In [ ]:
dask_controller.shutdown()

### Interactive Variant Overlay

In [ ]:
output_phenotype_kmyo_df = dd.read_parquet("/home/lag36T)D)/2022-07-01_lDE23_Final_Barcodes_df/",calculate_divisions=True)

In [ ]:
output_phenotype_kmyo_df.index

In [ ]:
headpath = "/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/Growth_Division/"
#note: shutdown dask when doing this...fix bug later
overlay_handle = tr.variant_overlay(headpath,"/home/de64/scratch/de64/sync_folder/2022-06-16_lDE23_Run_3/2022-07-01_lDE23_Final_Barcodes_df/",
                                 display_values_list=["Gene","TargetID","N Mismatch"],persist_data=False)  ##fix this, was improperly made (only initial cellID timepoints)

In [ ]:
overlay_handle.view_overlay(vmin=0,vmax=10000)

### Interactive Variant Table

In [ ]:
gene_table = dd.read_parquet("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/2022-09-08_lDE24_FInal_Barcodes_df_Run_1/",engine='pyarrow',calculate_divisions=True)
gene_table = gene_table.reset_index().set_index("phenotype trenchid", sorted=True)
gene_table = gene_table.groupby("phenotype trenchid",sort=False).apply(lambda x: x.iloc[0]).reset_index().set_index("FOV Parquet Index")
gene_table_out = gene_table.groupby("sgRNA").apply(lambda x: x.iloc[0])
gene_table_out["phenotype trenchids"] = gene_table.groupby("sgRNA").apply(lambda x: x["phenotype trenchid"].tolist())
gene_table_out = gene_table_out[["Gene",'Target Sequence',"phenotype trenchids","N Mismatch","N Target Sites","Category","Strand",]].compute()

In [ ]:
# Rerun from here
kymo_xarr = tr.kymo_xarr("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/")
wrapped_kymo_xarr = tr.kymo_xarr("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division",unwrap=False)

In [ ]:
gene_table_layout,select_gene,select_trenchid,select_unpacked_trenchid = tr.linked_gene_table(gene_table_out,trenchids_as_list=True,trenchid_column='phenotype trenchids')

In [ ]:
gene_table_layout

In [ ]:
output_display,save_button = tr.linked_kymograph_for_gene_table(kymo_xarr,wrapped_kymo_xarr,gene_table_out,select_gene,select_trenchid,select_unpacked_trenchid=select_unpacked_trenchid,\
                                                    trenchid_column='phenotype trenchids',y_scale=1.,x_window_size=1400)

In [ ]:
output_display

In [ ]:
save_button

In [ ]:
#rne: Prob. replication conflict resolution
#rplJ: Ribosome GTPase
#rpmJ: Nonessential ribosome component. Ribosome component that co-clusters with fts

# rho: Cells are very small and round, but their division looks good. I am not sure how they transition into a smaller size
# dnaA: cells with unfinished divisions 

# Wide cells seem to divide normally: (i) How do they become smaller? (ii) How 

# parC: Weird division formation. Very misplaced.'

#thrS: Long and then small

# FtsW: No reporter but division still happening 
# FtsI: aberrant midcell pos

In [ ]:
kymo_xarr

In [ ]:
wrapped_kymo_xarr